# Digital Trace Data - Lab 5: Digital Trace Data and AI

In this week's lab, we will explore how to apply machine learning (ML) techniques to digital trace data, focusing on text data.

This lab consists of four main parts:

1. Examining power relations in the use of Machine Learning (ML) in society:
    In this step, you will reflect on the benefits, biases and harms of the use of ML for predictive policing.
2. Using a ML model to Label Text.
    In this step, we will use a Natural Language Processing model (like chatGPT) to automatically label or categorize comments from Reddit. Transformer models are particularly powerful for understanding the context of text due to their ability to capture long-range dependencies between words. Specifically, you will:
* Input digital trace text data into the transformer model: We will use pre-trained models that have been fine-tuned for specific tasks, such as sentiment analysis, topic categorization, or keyword extraction.
* Generate predictions: The model will assign labels to the text based on its training. These labels will indicate hate (hate, not hate) or sentiment (positive, negative, neutral).
3. Reflecting on the origin of the biases on that ML model
4. Auditing the ML Model:
    Auditing a machine learning model is crucial to ensuring that it is fair, transparent, and reliable. In this step, we will evaluate where the model perform better or worse depending on the gender of the user commentingo on Reddit.
        
By the end of the lab, you will have hands-on experience applying state-of-the-art AI models to digital trace data and conducting an  audit of their performance and fairness.

# PART 1: Examining power
Predictive policing tools are increasingly used (including in the Netherlands). These systems are often trained on historical crime data and sometimes on personal data.* Note: The use of personal data for predictive policing has recently been banned in the EU.*

First watch the video in the classroom. Spend ~10 minutes discussing it in your groups before moving to the plenary discussion.


[![Predictive policing](https://img.youtube.com/vi/JkUrvZVbPeU/0.jpg)](https://www.youtube.com/watch?v=JkUrvZVbPeU)


### 1.1. Why could the ML models be biased/unfair?
* Sample bias: Is the training data representative, or does it mostly reflect over-policed communities?
* Outcome/label bias: Are we predicting “crime” or “arrests”? Do these labels capture reality fairly?
* Feature bias: Do variables (e.g., zip code, arrest history) act as proxies for race or class?
* Pipeline bias: Are the preprocessing, model design, or evaluation choices introducing skew?
* Application bias: How might the deployment of predictions reinforce existing inequalities?

### 1.2. Who is involved in creating predictive policing algorithms?
* Think about who is involved and their characteristics: police departments, governments, tech companies, citizens, and communities.

### 1.3. What are the assumptions underlying predictive policing? Whose interests are prioritized?
* Assumptions about the use of public money
* Whose goals are prioritized (e.g., efficiency of policing, state control)?
* Who might be overlooked or harmed (e.g., marginalized communities, people subject to surveillance)?

### 1.4. Who benefits from predictive policing:
* What potential benefits are claimed (crime prevention, resource allocation, efficiency)?
* What harms or risks come with biased predictions (reinforcing racism, loss of trust, over-policing)?

### 1.5. Do you think the benefits outweigh the biases, or not?

*WRITE ANSWER HERE*

## Part 2: Labeling data - Example of Hate Speech

Hate speech is defined
as a "direct and serious attack on any protected category of people based on their
race, ethnicity, national origin, religion, sex, gender, sexual orientation, disability
or disease" [[ref]](https://ojs.aaai.org/index.php/ICWSM/article/view/15041/14891).

In this part, we will analyze data from three subreddits and will try classifying their comments to check whether it is hate speech or not. The three subreddits are

- *dsa*: A subreddit dedicated to discussions related to the Democratic Socialists of America (DSA)
- *thenewright*: A subreddit centered around discussions of right-wing political ideologies
- *statistics*: A subreddit for discussions and questions related to the field of statistics


If you are interested in how we cleaned the data, this is [here](https://colab.research.google.com/github/digitalTraceData/digitalTraceData.github.io/blob/main/materials/lab5/lab5_clean_reddit_data.ipynb)

First, download the csv files for the three subreddits from the link written on the blackboard.

### 2.1 Open one of the csv files on your computer. Explain what the different variables (*author*, *body*, *controversiality*, *score*, and *created_utc*) mean? Which comment has the lowest/highest score?

Let's first load the files from these subreddits into Python. To do so, as done in the previous weeks, upload the files to Colab and replace the placeholders below by the actual paths.

In [ ]:
import pandas as pd

# Read the files
dsa = pd.read_csv("Placeholder", compression="gzip")
thenewright = pd.read_csv("Placeholder", compression="gzip")
statistics = pd.read_csv("Placeholder", compression="gzip")


# Showing what the data looks like
dsa.head()

*WRITE ANSWER HERE*

Next, we will use a machine learning model from [huggingface.co](https://huggingface.co/Hate-speech-CNERG/dehatebert-mono-english) to classify the content of the comments into hate-speech or non-hate-speech.

The code below first loads the pre-trained classification model from [huggingface.co](https://huggingface.co/Hate-speech-CNERG/dehatebert-mono-english) and then applies it to a sample of the data of a specified size (`sample_size` argument). The model requires the comments to be at most 512 characters, so we first truncate comments to their first 512 characters. The model provides a label (HATE/NON_HATE) and a probability of the label.

In [ ]:
from transformers import pipeline
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # suppress warning

def _pick_prob(all_scores, wanted_label_lower):
    # all_scores: list of dicts like [{"label": "negative", "score": 0.72}, ...]
    for d in all_scores:
        if d["label"].lower().startswith(wanted_label_lower):
            return float(d["score"])
    # fallback if labels are like LABEL_0, etc. (won't happen for Cardiff/CNERG)
    return 0.0

def classify_comment(pipe, comments, target="auto"):
    """
    Returns a probability in [0,1] suitable as a risk/negative score:
      - For Cardiff sentiment -> P(negative)
      - For hate-speech models -> P(HATE)
    """
    # Let the tokenizer handle length; batch for speed
    results = pipe(
        comments,
        top_k=None,
        truncation=True,
        padding=True,
        max_length=512,
        batch_size=32,
    )

    probs = []
    for rs in results:
        labels = [r["label"].lower() for r in rs]
        if target == "auto":
            # heuristics: if model has "negative", use that; else look for "hate"
            tgt = "negative" if "negative" in labels else "hate"
        else:
            tgt = target.lower()
        probs.append(_pick_prob(rs, tgt))
    return probs

def label_data(
    data,
    sample_size=1000,
    model_name="cardiffnlp/twitter-roberta-base-sentiment-latest",
    random_state=123,
):

    pipe = pipeline("text-classification", model=model_name, device=0)
    print(pipe.model.config.id2label)

    sample = data.sample(n=sample_size, replace=False, random_state=random_state).copy()
    # Choose target automatically based on model labels
    sample["hate_score"] = classify_comment(pipe, list(sample["body"]), target="auto")

    return sample


### 2.2 Apply the `label_data` function to the data from the three subreddits using the defaults.

Name them `df_tnr_label`, `df_stats_label`, `df_dsa_label`.

The code is written below, you could increase the sample_size but it will take a long time to run.


In [ ]:
# Labeling the data
df_tnr_label = label_data(thenewright, model_name="Hate-speech-CNERG/dehatebert-mono-english")
df_stats_label = label_data(statistics, model_name="Hate-speech-CNERG/dehatebert-mono-english")
df_dsa_label = label_data(dsa, model_name="Hate-speech-CNERG/dehatebert-mono-english")

In [ ]:
# Showing how the labeled data looks
df_tnr_label.head()

What is the comment with the highest hate score in each subreddit? (see code below for inspiration)

*WRITE ANSWER HERE*

In [ ]:
df_tnr_label.sort_values(by="hate_score", ascending=False)["body"].head().values

We can now analyze the provided hat_scores to the subreddits by plotting them.

What do the distributions tell you about the amount of hate on those platforms?

*WRITE ANSWER HERE*

In [ ]:
import seaborn as sns
import pylab as plt

sns.kdeplot(df_dsa_label["hate_score"], label="DSA")
sns.kdeplot(df_tnr_label["hate_score"], label="TheNewRight")
sns.kdeplot(df_stats_label["hate_score"], label="Statistics")
sns.despine()
plt.legend()


In [ ]:
# Convert 'created_utc' to datetime objects
df_tnr_label["year"] = pd.to_datetime(df_tnr_label["created_utc"], unit="s").dt.year
df_stats_label["year"] = pd.to_datetime(df_stats_label["created_utc"], unit="s").dt.year
df_dsa_label["year"] = pd.to_datetime(df_dsa_label["created_utc"], unit="s").dt.year

# Plot average hate score over time by subreddit
sns.lineplot(data=df_tnr_label, x="year", y="hate_score", label="TheNewRight")
sns.lineplot(data=df_stats_label, x="year", y="hate_score", label="Statistics")
sns.lineplot(data=df_dsa_label, x="year", y="hate_score", label="DSA")

plt.xlabel("Year")
plt.ylabel('Average Hate Score')
plt.legend()
sns.despine()
plt.show()


### 2.3 What is the approximate share of hate speech per subreddit according to the model?
Eyeball it from the figure

*WRITE ANSWER HERE*

### 2.4 Can you find an example labeled as hate speech that is likely not hate speech?  What do you think went wrong?

*WRITE ANSWER HERE*

### 2.5 For each subreddit, what is the correlation between `score` (as calculated in Reddit) and `hate_score` (the probability of a comment being hate speech, as inferred from the ML model).


*WRITE ANSWER HERE*

### 2.6 Now re-run the `label_data` function using [this](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) sentiment model (Hint: You can do this by setting `model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"` in the `label_data` function). What are the most negative comments? What are the most negative subreddits?

In [ ]:
# Labeling the data (code to be completed)
df_tnr_label_sent = label_data(thenewright, ...)

*WRITE ANSWER HERE*

### 2.7 For each subreddit, what is the correlation between `score` (as calculated in Reddit) and `hate_score` (the probability of a comment being negative, as inferred from the ML model)

*WRITE ANSWER HERE*

## Part 3: Reflection on AI Biases

### Data Collection and Annotation Bias

The hate speech model you used was trained using mostly data from Twitter and Stormfront (a neo-nazi forum) [[original source]](https://arxiv.org/pdf/2004.06465).



#### 3.1 Sample bias: to what extent is the data used in the training of the model representative of the Reddit data you are using? What are the potential implications for the analysis?

*WRITE ANSWER HERE*

#### 3.2 How could outcome bias (label bias) be affecting this model? (remember that the model was trained using several datasets from different sources and annotated in different ways).

*WRITE ANSWER HERE*

#### 3.3 If you use this model to label data and answer a research question, what type of measurment error would you be including?

*WRITE ANSWER HERE*

### Part 4: Auditing and correcting models:

We have now added three new columns: `gender` (male/female/queer/neutral), `label_value` (the "true" label of hate) and `score` (the label of hate predicted by the previous model, where a score greater than 0.5 is considered hate). Assume the `gender` and `label_value` columns have no mistakes for the sake of the exercise.

Download the data set `df_dsa_label_true.csv` with these columns from the link above and upload them into Colab. Then load it into Python with the code below:

In [ ]:
# Load artificially labelled dsa comments
import pandas as pd
df_dsa_label_true = pd.read_csv("df_dsa_label_true.csv.gz", compression="gzip",index_col=0)
df_dsa_label_true.head()

### 4.1 Based on the confusion matrix below (columns = predicted by algorithm; row = true values). Is the model good?

*WRITE ANSWER HERE*


In [ ]:
print("All groups")
pd.crosstab(df_dsa_label_true["label_value"], df_dsa_label_true["score"])


### 4.2 Based on the confusion matrixes below per group
* What group has a higher false positive rate (false positives / negatives)?

*WRITE ANSWER HERE*

* What group has a higher false negative rate (false negative / positives)?

*WRITE ANSWER HERE*

In [ ]:
for group, df_group in df_dsa_label_true.groupby("gender"):
  print(group)
  display(pd.crosstab(df_group["label_value"], df_group["score"]))

Next, we can use the `fairlearn` package to audit the model.

Here is an example of how to audit a model using several fairness metrics


In [ ]:
!pip install fairlearn

In [ ]:
from sklearn.metrics import confusion_matrix
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    true_negative_rate,
    false_positive_rate,
    false_negative_rate
)
import pylab as plt

# --- False discovery and omission rates ---
def false_discovery_rate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tp) if (fp + tp) > 0 else 0.0

def false_omission_rate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fn / (fn + tn) if (fn + tn) > 0 else 0.0

 # we'll remove females since there are few cases and it complicates the interpretation, don't do it in real data!
df = df_dsa_label_true.loc[df_dsa_label_true["gender"] != "female"]

# For an audit, we need the true labels, the sensitive attribute(s), and the predicted labels
y_true = df["label_value"]      # true labels (0/1)
sensitive = df["gender"]        # sensitive attribute
y_pred = df["score"]            # model predictions (0/1)

# We also need some metrics
metrics = {
    "Selection Rate": selection_rate,         # P(ŷ = 1)

    # Punitive example
    "FPR": false_positive_rate,               # P(ŷ = 1 | y = 0) FP/not cirminal
    "FDR": false_discovery_rate,              # P(y = 0 | ŷ = 1) FP/predicted criminal

    # Assistive example
    "FNR": false_negative_rate,               # P(ŷ = 0 | y = 1) FN / in need
    "FOR": false_omission_rate,               # P(y = 1 | ŷ = 0) FN / predicted not in need
    "TPR (Recall)": true_positive_rate,       # P(ŷ = 1 | y = 1) # TP / in need --> if resources are limited
}


In [ ]:

# Set up the audit
mf = MetricFrame(metrics=metrics, y_true=y_true, y_pred=y_pred, sensitive_features=sensitive)

# Print the audit
print("Metrics by group:\n", mf.by_group, "\n")

# quick bar charts (one per metric)
mf.by_group.plot(
    kind="barh", subplots=True, layout=(3,3), figsize=(8,5), title="Fairness metrics by gender", sharex=False, legend=False,
)
plt.tight_layout()

### 4.3 Imagine you are using the ML model for an assistive intervention: you want to detect hate speech and offer extra holidays and mental health resources to those people. Which metric would you use to assess fairness? Is the model above  fair?

Hint: see the [decision tree](https://www.datasciencepublicpolicy.org/our-work/tools-guides/aequitas/).

*WRITE ANSWER HERE*

### 4.4 Imagine you are using the ML model for a punitive intervention: you want to detect hate speech and fine those people. Which metric would you use to assess fairness? Is the model above fair?

Hint: see the [decision tree](https://www.datasciencepublicpolicy.org/our-work/tools-guides/aequitas/).

*WRITE ANSWER HERE*

### 4.5. What information do you need to be able to correct your model? Do you have that information in your group project?


*WRITE ANSWER HERE*

### Are you still interested in how to audit and mitigate errors in models?
Take a look at these resources:
- https://fairlearn.org/
- https://github.com/microsoft/responsible-ai-toolbox